# 저해상 청구기호 전용 OCR 인식기 파인튜닝

**목적:** 대림 광각 사진처럼 뭉개진 라벨 텍스트(줄높이 7~16px)를 읽도록
`korean_PP-OCRv5_mobile_rec`를 **합성 저해상 라벨 5,883쌍**으로 파인튜닝.

**업로드 파일:** `synth_rec.zip`

**사용법:** GPU(T4) 런타임 → 셀 순서대로. (paddle 설치 후 **세션 재시작** 필수)

⚠️ PaddleOCR 버전에 따라 config/사전학습 경로가 다를 수 있어 자동 탐색 셀을 넣어뒀습니다.
실패하는 셀이 있으면 출력 메시지와 함께 알려주세요.

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 데이터 업로드 + 배치
from google.colab import files
up = files.upload()   # synth_rec.zip
!unzip -oq synth_rec.zip -d PaddleOCR/train_data/
!ls PaddleOCR/train_data/synth_rec/train | head -3
!head -2 PaddleOCR/train_data/synth_rec/train/rec_gt_train.txt

In [ ]:
# 3) config·사전학습 모델 자동 탐색
%cd /content
import glob, os
cfgs = glob.glob('PaddleOCR/configs/rec/**/*korean*', recursive=True)
print('korean rec configs:', cfgs)
CFG = next((c for c in cfgs if 'v5' in c.lower() and 'mobile' in c.lower()), cfgs[0] if cfgs else None)
print('사용 config:', CFG)
# 사전학습 가중치 후보 URL (버전에 따라 하나만 유효)
urls = [
 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
 'https://paddleocr.bj.bcebos.com/PP-OCRv5/multilingual/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
]
for u in urls:
    if os.system(f'wget -q {u} -O pretrain.pdparams') == 0 and os.path.getsize('pretrain.pdparams') > 1e6:
        print('✅ 사전학습 확보:', u); break
else:
    print('❌ 사전학습 다운로드 실패 — 이 메시지를 공유해주세요')

In [ ]:
# 4) 학습 (15 epochs, T4 기준 ~40-60분)
# 주의: 이 config의 실제 배치는 loader가 아니라 Train.sampler.first_bs가 결정함 (MultiScaleSampler)
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/train.py -c {CFG_REL} \
  -o Global.pretrained_model=/content/pretrain \
     Global.epoch_num=15 \
     Global.save_model_dir=./output/korean_lowres \
     Global.eval_batch_step="[0,500]" \
     Optimizer.lr.learning_rate=0.0001 \
     Train.dataset.data_dir=./train_data/synth_rec/train \
     Train.dataset.label_file_list=["./train_data/synth_rec/train/rec_gt_train.txt"] \
     Eval.dataset.data_dir=./train_data/synth_rec/val \
     Eval.dataset.label_file_list=["./train_data/synth_rec/val/rec_gt_val.txt"] \
     Train.sampler.first_bs=32 \
     Train.loader.batch_size_per_card=32 \
     Eval.loader.batch_size_per_card=32

In [ ]:
# 5) 추론 모델로 내보내기 + 다운로드
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/export_model.py -c {CFG_REL} \
  -o Global.pretrained_model=./output/korean_lowres/best_accuracy \
     Global.save_inference_dir=./korean_lowres_rec_infer
!zip -q -r /content/korean_lowres_rec_infer.zip korean_lowres_rec_infer
from google.colab import files
files.download('/content/korean_lowres_rec_infer.zip')
print('로컬 적용: PaddleOCR(text_recognition_model_dir="korean_lowres_rec_infer", ...)')